In [1]:
import importlib.util, subprocess, sys

_pkgs = ["langgraph", "langchain_community", "langchain_openai", "langsmith", "grandalf"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

In [2]:
# Environment Variable Initialization

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from the notebook's working directory

def _set_if_undefined(var_name: str):
    value = os.environ.get(var_name, "").strip()
    if value:
        masked = value[:6] + "*" * (min(20, len(value) - 6))
        print(f"  ✅ {var_name}: {masked}")
    else:
        print(f"  ❌ {var_name}: not set")

# ---- Environment Variables Required ----

print("Checking environment variables...")
_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("LANGSMITH_TRACING") 
_set_if_undefined("LANGSMITH_API_KEY") # https://docs.langchain.com/langsmith/observability
_set_if_undefined("MODEL")
print("Done.")

Checking environment variables...
  ✅ OPENAI_API_KEY: sk-pro********************
  ✅ LANGSMITH_TRACING: true
  ✅ LANGSMITH_API_KEY: lsv2_p********************
  ✅ MODEL: gpt-4o*****
Done.


In [3]:
# Lift Controller with LangGraph:
# - A multi-step, multi-agent workflow for managing elevator (lift) requests.
# - The controller determines the closest lift and routes it to the requesting floor.
# - Steps include requesting location, selecting lift, and dispatching it.

# ---- Imports ----

import os
from typing import Literal
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.types import Command

# ---- LLM Setup ----

# Load model from environment
default_model = os.environ["MODEL"]

# Initialize ChatOpenAI
llm = ChatOpenAI(model=default_model)

# ---- Configuration ----

lifts = ["lift1", "lift2"]
floors = ["floor1", "floor2", "floor3", "floor4", "floor5"]
options = lifts + [END]

# ---- State Definitions ----

class OverallState(MessagesState):
    """Graph state with current step and selected lift."""
    currentstep: Literal["1", "2", "3", END]
    selectedlift: Literal[*lifts]

class NextStep(TypedDict):
    """Output format for the controller to decide next step."""
    step: Literal["1", "2", "3", END]

class SelectedLift(TypedDict):
    """Output format for the selected lift."""
    lift: Literal[*lifts]

class LiftLocations(TypedDict):
    """Current floor location of each lift."""
    lift1loc: str
    lift2loc: str

# ---- System Prompt ----

controller_system_prompt = (
    f"You are a lift controller managing a building with {len(floors)} floors and {len(lifts)} lifts. "
    "Each floor has a call button that summons a lift. Lifts are stationed at various floors. "
    "Your goal is to choose the most appropriate lift based on proximity.\n\n"

    "# Instructions:\n"
    "1. Ask all lifts to report their current location.\n"
    "2. Analyze their positions and determine which is closest to the requesting floor. If there is a tie choose any one.\n"
    "3. Send a reservation to the selected lift and rejection notices to the others.\n\n"

    "# Example:\n"
    "- If Lift1 is on floor 1 and Lift2 on floor 5, and the user is on floor 2:\n"
    "- Lift1 is closer (|2 - 1| = 1 vs |2 - 5| = 3).\n"
    "- So, select Lift1.\n\n"

    "Always follow these steps precisely and never skip to the next step without completion of the previous one.\n"
    "Now, analyze the message history to determine the next step."
)

# ---- Controller Node ----

def lift_controller_node(state: OverallState) -> Command[Literal[*lifts, END]]:
    """Main controller logic: decide next step and route accordingly."""
    messages = [
        {"role": "system", "content": controller_system_prompt},
    ] + state["messages"] + [
        "Based on the message history, and the lift request handling process, what should be the next step? make sure all 3 steps are executed. If complete, respond with -1."
    ]

    response = llm.with_structured_output(NextStep).invoke(messages) # https://docs.langchain.com/oss/python/langchain/overview
    nextstep = response["step"]

    if nextstep == "1":
        print("************** Step 1: Request Locations ***************")
        return Command(
            goto=["lift1", "lift2"],
            update={
                "messages": [HumanMessage(content="Step 1 -> CFP: Share your current location", name="lift_controller")],
                "currentstep": nextstep
            }
        )

    elif nextstep == "2":
        print("************** Step 2: Analyze and Select Lift ***************")

        loc_request = [
            {"role": "system", "content": controller_system_prompt}
        ] + state["messages"] + [
            "Based on the message history, identify lift1 and lift2 current locations."
        ]

        locations = llm.with_structured_output(LiftLocations).invoke(loc_request)
        lift1loc, lift2loc = locations["lift1loc"], locations["lift2loc"]
        print("Lift locations:", locations)

        reasoning_request = state["messages"] + [
            f"Given lift1 is at {lift1loc} and lift2 is at {lift2loc}, which is closer to the user's floor?"
        ]
        reasoning = llm.invoke(reasoning_request).content

        selected_lift = llm.with_structured_output(SelectedLift).invoke(reasoning_request)["lift"]
        print("Selected lift:", selected_lift)

        return Command(
            goto=["lift1", "lift2"],
            update={
                "messages": [HumanMessage(content=f"Step 2 -> Reasoning: {reasoning} Selected lift: {selected_lift}", name="lift_controller")],
                "currentstep": nextstep
            }
        )

    elif nextstep == "3":
        print("************** Step 3: Dispatch Selected Lift ***************")
        selected_lift = llm.with_structured_output(SelectedLift).invoke(messages)["lift"]
        return Command(
            goto=[selected_lift],
            update={
                "messages": [HumanMessage(content=f"Step 3 -> Selection: You ({selected_lift}) are selected", name="lift_controller")],
                "currentstep": nextstep
            }
        )

    print("************** END ***************")
    return Command(goto=END)

In [4]:
# ---- Lift Agent Nodes ----

def lift1_node(state: OverallState) -> Command[Literal["lift_controller"]]:
    """Behavior of Lift1 based on current step."""
    response_map = {
        "1": "1",  # Location
        "2": "Acknowledge",
        "3": "Moving to target floor"
    }
    return Command(
        update={
            "messages": [HumanMessage(content=response_map[state["currentstep"]], name="lift1")]
        },
        goto="lift_controller",
    )

def lift2_node(state: OverallState) -> Command[Literal["lift_controller"]]:
    """Behavior of Lift2 based on current step."""
    response_map = {
        "1": "5",  # Location
        "2": "Acknowledge",
        "3": "Moving to target floor"
    }
    return Command(
        update={
            "messages": [HumanMessage(content=response_map[state["currentstep"]], name="lift2")]
        },
        goto="lift_controller",
    )

In [5]:
# ---- Graph Assembly ----

builder = StateGraph(MessagesState)
builder.set_entry_point("lift_controller")
builder.add_node("lift_controller", lift_controller_node)
builder.add_node("lift1", lift1_node)
builder.add_node("lift2", lift2_node)

graph = builder.compile()

In [6]:
# ---- Visualization ----
graph.get_graph().print_ascii()

                +-----------+                   
                | __start__ |                   
                +-----------+                   
                       *                        
                       *                        
                       *                        
              +-----------------+               
              | lift_controller |               
              +-----------------+               
             ...       .       ...              
          ...          .          ...           
        ..             .             ..         
+-------+         +-------+         +---------+ 
| lift1 |         | lift2 |         | __end__ | 
+-------+         +-------+         +---------+ 


In [7]:
for s in graph.stream(
    {"messages": [("user", "floor 4")]}, debug=True):
    print(s)
    print("============================")

[values] {'messages': [HumanMessage(content='floor 4', additional_kwargs={}, response_metadata={}, id='12f1e3a2-45d7-4ec4-bef9-44ded74b8d08')]}
************** Step 1: Request Locations ***************
[updates] {'lift_controller': {'messages': [HumanMessage(content='Step 1 -> CFP: Share your current location', additional_kwargs={}, response_metadata={}, name='lift_controller', id='3c191d80-df7a-4473-9f11-7b07a18104df')], 'currentstep': '1'}}
{'lift_controller': {'messages': [HumanMessage(content='Step 1 -> CFP: Share your current location', additional_kwargs={}, response_metadata={}, name='lift_controller', id='3c191d80-df7a-4473-9f11-7b07a18104df')], 'currentstep': '1'}}
[values] {'messages': [HumanMessage(content='floor 4', additional_kwargs={}, response_metadata={}, id='12f1e3a2-45d7-4ec4-bef9-44ded74b8d08'), HumanMessage(content='Step 1 -> CFP: Share your current location', additional_kwargs={}, response_metadata={}, name='lift_controller', id='3c191d80-df7a-4473-9f11-7b07a18104df'

KeyError: 'step'